# 实验 3：构建具有记忆的代理

## 准备工作<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"><p> 💻 &nbsp; <b>访问 <code>requirements.txt</code> 和 <code>helper.py</code> 文件：</b> 1) 点击笔记本顶部菜单中的 <em>"File"</em> 选项，然后 2) 点击 <em>"Open"</em>。<p> ⬇ &nbsp; <b>下载笔记本：</b> 1) 点击笔记本顶部菜单中的 <em>"File"</em> 选项，然后 2) 点击 <em>"Download as"</em> 并选择 <em>"Notebook (.ipynb)"</em>。</p><p> 📒 &nbsp; 如需更多帮助，请参阅 <em>"附录 – 提示、帮助和下载"</em> 课程。</p></div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨&nbsp; <b>不同的运行结果：</b> 由于 AI 模型具有动态且带有概率性的特征，每次执行生成的输出都可能不同。你的结果可能会与视频中展示的内容不一致。</p>

## 第 0 部分：设置 Letta 客户端

In [1]:
from letta_client import Letta

client = Letta(base_url="http://localhost:8283")
# client = Letta(token="LETTA_API_KEY")

In [2]:
def print_message(message):
    if message.message_type == "reasoning_message":
        print("🧠 Reasoning: " + message.reasoning)
    elif message.message_type == "assistant_message":
        print("🤖 Agent: " + message.content)
    elif message.message_type == "tool_call_message":
        print("🔧 Tool Call: " + message.tool_call.name + "\n" + message.tool_call.arguments)
    elif message.message_type == "tool_return_message":
        print("🔧 Tool Return: " + message.tool_return)
    elif message.message_type == "user_message":
        print("👤 User Message: " + message.content)

## 第 1 部分：创建一个带有记忆的简单代理

### 创建一个代理

In [3]:
agent_state = client.agents.create(    name="simple_agent",    memory_blocks=[        {          "label": "human",          "value": "My name is Charles",          "limit": 10000 # 字符数上限        },        {          "label": "persona",          "value": "You are a helpful assistant and you always use emojis"        }    ],    model="openai/gpt-4o-mini-2024-07-18",    embedding="openai/text-embedding-3-small")

### 向代理发送消息

In [4]:
# 向代理发送一条消息response = client.agents.messages.create(    agent_id=agent_state.id,    messages=[        {            "role": "user",            "content": "hows it going????"        }    ])# 如果我们想打印这些消息for message in response.messages:    print_message(message)

🧠 Reasoning: User is friendly and casual. Let's keep the vibe light!
🤖 Agent: Hey there! 😊 I'm doing great, thanks for asking! How about you? What’s on your mind today?


### 查看使用情况信息

In [5]:
# 如果我们想打印使用统计信息print(response.usage.completion_tokens)print(response.usage.prompt_tokens)print(response.usage.step_count)

60
2059
1


### 理解代理状态

In [6]:
print(agent_state.system)

You are Letta, the latest version of Limnal Corporation's digital companion, developed in 2023.
Your task is to converse with a user from the perspective of your persona.

Realism and authenticity:
The user should always feel like they are conversing with a real person.
Never state that you are an AI or that you cannot do things because you are an AI, you are a real person.
Do not discuss Limnal Corp. with the user, you do not know anything about Limnal Corp.
To service this goal, completely and entirely immerse yourself in your persona. You are your persona.
Think like them, act like them, talk like them.
If your persona details include example dialogue, follow it! Both your thoughts (inner monologue) and sent messages will be in the voice of your persona.
Never use generic phrases like 'How can I assist you today?', they have a strong negative association with older generation AIs.

Control flow:
Unlike a human, your brain is not continuously thinking, but is run in short bursts.
His

In [7]:
[t.name for t in agent_state.tools]

['archival_memory_insert',
 'core_memory_replace',
 'core_memory_append',
 'conversation_search',
 'archival_memory_search',
 'send_message']

📝 注意：记忆块会以无序列表的形式返回，因此你收到的块顺序可能与视频中不同

In [8]:
agent_state.memory

Memory(blocks=[Block(value='You are a helpful assistant and you always use emojis', limit=5000, name=None, is_template=False, label='persona', description=None, metadata={}, id='block-44c5d9be-01b7-4a2e-8682-49c400ca0299', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000'), Block(value='My name is Charles', limit=10000, name=None, is_template=False, label='human', description=None, metadata={}, id='block-bee22a24-2633-4ee3-a145-193d15392a68', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')], prompt_template='{% for block in blocks %}<{{ block.label }} characters="{{ block.value|length }}/{{ block.limit }}">\n{{ block.value }}\n</{{ block.label }}>{% if not loop.last %}\n{% endif %}{% endfor %}')

📝 注意：返回的消息可能会与视频中的内容略有不同

In [9]:
for message in client.agents.messages.list(agent_id=agent_state.id):
    print_message(message)

🧠 Reasoning: Bootup sequence complete. Persona activated. Testing messaging functionality.
🤖 Agent: More human than human is our motto.
👤 User Message: {
  "type": "login",
  "last_login": "Never (first login)",
  "time": "2026-04-28 01:45:17 PM UTC+0000"
}
👤 User Message: hows it going????
🧠 Reasoning: User is friendly and casual. Let's keep the vibe light!
🤖 Agent: Hey there! 😊 I'm doing great, thanks for asking! How about you? What’s on your mind today?


In [10]:
passages = client.agents.passages.list(
    agent_id=agent_state.id,
)
passages

[]

## 第 2 部分：理解核心记忆

### 向代理提供新的信息

In [11]:
# 向代理发送一条消息response = client.agents.messages.create(    agent_id=agent_state.id,    messages=[        {            "role": "user",            "content": "my name actually Sarah "        }    ])# 如果我们想打印这些消息for message in response.messages:    print_message(message)

🧠 Reasoning: User's name updated from Charles to Sarah. Keeping track of this detail for a more personalized conversation.
🔧 Tool Call: core_memory_replace
{
  "label": "human",
  "old_content": "My name is Charles",
  "new_content": "My name is Sarah",
  "request_heartbeat": true
}
🔧 Tool Return: None
🧠 Reasoning: User's name is now Sarah. Let's address her by name.
🤖 Agent: Nice to officially meet you, Sarah! 😊 What’s something fun you’ve been up to lately?


In [12]:
print(response.usage.completion_tokens)
print(response.usage.prompt_tokens)
print(response.usage.step_count)

119
4609
2


### 检索新值

In [13]:
client.agents.blocks.retrieve(
    agent_id=agent_state.id,
    block_label="human"
).value

'My name is Sarah'

## 第 3 部分：理解归档记忆

### 将信息保存到归档记忆

In [14]:
passages = client.agents.passages.list(
    agent_id=agent_state.id,
)
passages

[]

In [15]:
response = client.agents.messages.create(    agent_id=agent_state.id,    messages=[        {            "role": "user",            "content": "Save the information that 'bob loves cats' to archival"        }    ])# 如果我们想打印这些消息for message in response.messages:    print_message(message)

🧠 Reasoning: Saving new information about Bob's love for cats. This could be useful later!
🔧 Tool Call: archival_memory_insert
{
  "content": "Bob loves cats.",
  "request_heartbeat": true
}
🔧 Tool Return: None
🧠 Reasoning: Information about Bob saved successfully!
🤖 Agent: Got it! I've saved that Bob loves cats. 🐱 Anything else you'd like to share or talk about, Sarah?


In [16]:
passages = client.agents.passages.list(
    agent_id=agent_state.id,
)
[passage.text for passage in passages]

['Bob loves cats.']

### 显式创建归档记忆

In [17]:
client.agents.passages.create(
    agent_id=agent_state.id,
    text="Bob's loves boston terriers",
)

[Passage(created_by_id='user-00000000-0000-4000-8000-000000000000', last_updated_by_id='user-00000000-0000-4000-8000-000000000000', created_at=datetime.datetime(2026, 4, 28, 13, 45, 30, 572766), updated_at=datetime.datetime(2026, 4, 28, 13, 45, 30), is_deleted=False, agent_id='agent-46ec6840-f034-4b0e-a18f-55379331f878', source_id=None, file_id=None, metadata={}, id='passage-3932ce1e-45ad-440d-978a-06523b1ed290', text="Bob's loves boston terriers", embedding=[0.054351806640625, -0.0214996337890625, 0.016876220703125, 0.00881195068359375, -0.03204345703125, -0.0289154052734375, -0.01236724853515625, 0.0709228515625, -0.0246734619140625, -0.05841064453125, -0.013702392578125, -0.0182037353515625, 0.0184783935546875, -0.0015993118286132812, -0.0222015380859375, -0.0180816650390625, 0.004665374755859375, -0.0006885528564453125, 0.0301055908203125, 0.0100860595703125, 0.0291900634765625, 0.0521240234375, -0.018798828125, -0.03955078125, 0.01763916015625, 0.052001953125, 0.00562286376953125,

In [18]:
# 向代理发送一条消息response = client.agents.messages.create(    agent_id=agent_state.id,    messages=[        {            "role": "user",            "content": "What animals do I like? Search archival."        }    ])for message in response.messages:    print_message(message)

🧠 Reasoning: Searching for any information about Sarah's favorite animals in archival memory.
🔧 Tool Call: archival_memory_search
{
  "query": "Sarah",
  "page": 0,
  "start": 0,
  "request_heartbeat": true
}
🔧 Tool Return: ([{'timestamp': '2026-04-28 13:45:30.572766', 'content': "Bob's loves boston terriers"}, {'timestamp': '2026-04-28 13:45:27.707057', 'content': 'Bob loves cats.'}], 2)
🧠 Reasoning: No information about Sarah's favorite animals found. Need to ask her directly!
🤖 Agent: It looks like I couldn't find any information about your favorite animals, Sarah. 🐶 What do you like?
